# Laboratorio: del HTML al JSON del contrato

Este notebook explica de forma **simple y lineal** el pipeline del laboratorio (captura → limpieza → Gemini → JSON).

En el notebook `01_obtener_y_limpiar_noticia.ipynb` se llegó a un diccionario `{url, titulo, texto}`.

Aquí damos un paso más:

1. Descargar una noticia (HTML).
2. Extraer título y párrafos.
3. Limpiar el texto.
4. Enviar el texto a **Gemini** para obtener el JSON del laboratorio.
5. Validar ese JSON (el LLM no es la fuente de verdad).
6. Guardar el archivo intermedio **antes** de convertirlo a notas de Obsidian.

No se genera el vault de Obsidian en este notebook. Ese paso es la siguiente etapa del laboratorio (`TODO(alumno)` en el repositorio).

> **Importante:** revise los términos de uso y el `robots.txt` de cada medio. Extraiga solo información **explícita** en la noticia: no invente imputados, delitos ni relaciones.


## 1. Instalar e importar las librerías

Usaremos:

- `requests`: descargar la página.
- `BeautifulSoup`: recorrer el HTML.
- `re`: limpieza básica del texto.
- `google-genai`: llamada a Gemini (igual que en el proyecto).


In [ ]:
!pip -q install beautifulsoup4 requests google-genai


In [ ]:
import json
import os
import re

from google import genai
from google.genai import types
import requests
from bs4 import BeautifulSoup


## 2. Indicar la URL de una noticia

Cambie la siguiente URL por una noticia que quiera analizar.

Se usa un `User-Agent` de navegador porque algunos sitios rechazan peticiones sin identificar el cliente.


In [ ]:
# URL de una noticia real publicada en Cooperativa (la misma del notebook 01).
url = "https://www.cooperativa.cl/noticias/site/artic/20260902/pags-amp/20260902074532.html"

# Identificador estable: en el proyecto recorre HTML → texto → JSON → nota Markdown.
id_noticia = "N001"
fuente = "Cooperativa"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers, timeout=15)

print("Código HTTP:", response.status_code)
print("Cantidad de caracteres descargados:", len(response.text))


## 3. Interpretar el HTML con BeautifulSoup

Una página web contiene muchas etiquetas:

```html
<h1>Título de la noticia</h1>
<p>Primer párrafo...</p>
<p>Segundo párrafo...</p>
```

BeautifulSoup transforma el HTML en una estructura que Python puede recorrer.


In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

print(soup.prettify()[:2000])


## 4. Extraer el título

En muchos sitios el título principal está dentro de una etiqueta `<h1>`.


In [ ]:
h1 = soup.find("h1")

if h1:
    titulo = h1.get_text(" ", strip=True)
else:
    titulo = "Título no encontrado"

print(titulo)


## 5. Extraer los párrafos

La estrategia más simple es recuperar las etiquetas `<p>` y descartar textos cortos (menús, firmas, botones).

Si existe `<article>`, se prefiere ese bloque: suele ser el cuerpo de la noticia y evita publicidad o recomendaciones.


In [ ]:
LARGO_MINIMO = 40  # igual criterio que LimpiadorHTML en el repositorio


def extraer_parrafos(nodo):
    encontrados = []
    for p in nodo.find_all("p"):
        texto = p.get_text(" ", strip=True)
        if len(texto) >= LARGO_MINIMO:
            encontrados.append(texto)
    return encontrados


article = soup.find("article")
if article:
    parrafos = extraer_parrafos(article)
    origen = "article"
else:
    parrafos = extraer_parrafos(soup)
    origen = "pagina completa"

print(f"Origen de los párrafos: {origen}")
print("Párrafos encontrados:", len(parrafos))

for i, p in enumerate(parrafos[:5], start=1):
    print(f"\nPárrafo {i}:")
    print(p)


## 6. Construir el texto completo de la noticia


In [ ]:
texto_noticia = "\n".join(parrafos)

print(texto_noticia[:4000])


## 7. Limpieza básica del texto

En esta etapa eliminaremos:

- espacios repetidos;
- saltos de línea excesivos;
- tabulaciones;
- espacios antes de signos de puntuación.

No se eliminan palabras ni se cambia el sentido de la noticia. El LLM debe trabajar con el texto explícito, no con una versión “mejorada”.


In [ ]:
def limpiar_texto(texto):
    texto = texto.replace("\t", " ")
    texto = re.sub(r"[ ]+", " ", texto)
    texto = re.sub(r"\n\s*\n+", "\n", texto)
    texto = re.sub(r"\s+([,.;:!?])", r"\1", texto)
    return texto.strip()


texto_limpio = limpiar_texto(texto_noticia)

print(texto_limpio[:4000])


## 8. Estructura intermedia (antes de Gemini)

Todavía no hace falta una base de datos.

Este diccionario es el análogo simple de `NoticiaFuente` en el proyecto: identifica la pieza y lleva el texto limpio que se enviará al modelo.


In [ ]:
noticia = {
    "id_noticia": id_noticia,
    "fuente": fuente,
    "url": url,
    "titulo": titulo,
    "texto": texto_limpio,
}

noticia


## 9. Contrato JSON del laboratorio

Gemini no debe devolver un resumen libre. Debe devolver **exactamente** estos campos:

| Campo | Tipo | Contenido |
| --- | --- | --- |
| `id_noticia` | texto | Clave estable (`N001`, …) |
| `titulo` | texto o `null` | Título de la pieza |
| `fecha_publicacion` | texto o `null` | Fecha si aparece |
| `fuente`, `url` | texto | Medio y enlace |
| `resumen` | texto o `null` | Síntesis breve, sin inventar |
| `delitos` | lista de textos | Tipos delictuales explícitos |
| `personas` | lista de `{nombre, rol}` | Solo personas nombradas o referidas |
| `organizaciones` | lista de textos | Policía, fiscalía, bandas, etc. |
| `lugares` | lista de textos | Comuna, ciudad, región |
| `objetos` | lista de `{tipo, nombre, cantidad, unidad}` | Armas, sustancias, vehículos |
| `relaciones` | lista de `{origen, tipo, destino}` | Enlaces explícitos |

**Principio:** si un dato no está en el texto, use `null` o una lista vacía. Los roles (detenido, imputado, víctima, testigo) no son equivalentes.


In [ ]:
CAMPOS_OBLIGATORIOS = [
    "id_noticia",
    "titulo",
    "fecha_publicacion",
    "fuente",
    "url",
    "resumen",
    "delitos",
    "personas",
    "organizaciones",
    "lugares",
    "objetos",
    "relaciones",
]

CAMPOS_LISTA = [
    "delitos",
    "personas",
    "organizaciones",
    "lugares",
    "objetos",
    "relaciones",
]

print("Campos obligatorios:")
for campo in CAMPOS_OBLIGATORIOS:
    print("-", campo)


## 10. Extraer el JSON con Gemini

En Google Colab, guarde la clave en **Secrets** con el nombre `GEMINI_API_KEY` (icono de llave a la izquierda). **Nunca** la escriba en una celda ni la suba a GitHub.

Si no hay clave o la API falla, el notebook usa un JSON de ejemplo con el mismo esquema para que el resto del flujo se pueda ejecutar.


In [ ]:
MODELO_GEMINI = "gemini-2.0-flash"

# JSON de respaldo (mismo contrato). Permite continuar sin API.
JSON_EJEMPLO = {
    "id_noticia": id_noticia,
    "titulo": titulo,
    "fecha_publicacion": None,
    "fuente": fuente,
    "url": url,
    "resumen": (
        "Carabineros detuvo en La Reina a un hombre de 42 años que se hacía "
        "pasar por runner para asaltar a adolescentes y quitarles sus celulares."
    ),
    "delitos": ["robo con violencia", "robo con intimidacion"],
    "personas": [
        {"nombre": "Carolina Constanzo", "rol": "oficial de Carabineros"},
        {"nombre": "hombre de 42 anos", "rol": "detenido"},
    ],
    "organizaciones": ["Carabineros", "Fiscalia", "16 Comisaria de La Reina"],
    "lugares": ["La Reina"],
    "objetos": [
        {"tipo": "arma", "nombre": "arma blanca", "cantidad": None, "unidad": None},
        {"tipo": "especie", "nombre": "telefono celular", "cantidad": None, "unidad": None},
    ],
    "relaciones": [
        {
            "origen": "hombre de 42 anos",
            "tipo": "DETENIDO_EN",
            "destino": "La Reina",
        },
        {
            "origen": "hombre de 42 anos",
            "tipo": "INVESTIGADO_POR",
            "destino": "robo con violencia",
        },
    ],
}


def obtener_api_key():
    """Lee GEMINI_API_KEY desde Colab Secrets o desde el entorno local."""
    try:
        from google.colab import userdata

        clave = userdata.get("GEMINI_API_KEY")
        if clave:
            return clave.strip()
    except Exception:
        pass
    return os.environ.get("GEMINI_API_KEY", "").strip()


def construir_prompt(noticia_fuente):
    campos = ", ".join(CAMPOS_OBLIGATORIOS)
    return (
        "Analiza la siguiente noticia delictual.\n\n"
        "Extrae solamente informacion explicita. No inventes datos, "
        "entidades, roles ni relaciones.\n"
        "Devuelve exclusivamente JSON valido, sin markdown ni explicaciones.\n\n"
        f"Campos obligatorios: {campos}.\n"
        "personas: lista de objetos con claves nombre y rol.\n"
        "objetos: lista de objetos con claves tipo, nombre, cantidad, unidad.\n"
        "relaciones: lista de objetos con claves origen, tipo, destino.\n"
        "Si un dato no aparece, usa null o una lista vacia.\n\n"
        f"id_noticia: {noticia_fuente['id_noticia']}\n"
        f"fuente: {noticia_fuente['fuente']}\n"
        f"url: {noticia_fuente['url']}\n\n"
        "NOTICIA:\n"
        f"{noticia_fuente['texto']}\n"
    )


def parsear_json_gemini(bruto):
    texto = (bruto or "").strip()
    cerca = re.search(r"```(?:json)?\s*(.*?)\s*```", texto, re.DOTALL)
    if cerca:
        texto = cerca.group(1)
    data = json.loads(texto)
    if not isinstance(data, dict):
        raise ValueError("La respuesta de Gemini no es un objeto JSON.")
    return data


def extraer_con_gemini(noticia_fuente):
    clave = obtener_api_key()
    if not clave:
        raise RuntimeError(
            "Falta GEMINI_API_KEY. En Colab: Secrets → GEMINI_API_KEY. "
            "Nunca escriba la clave en el notebook."
        )
    cliente = genai.Client(api_key=clave)
    respuesta = cliente.models.generate_content(
        model=MODELO_GEMINI,
        contents=construir_prompt(noticia_fuente),
        config=types.GenerateContentConfig(
            automatic_function_calling=types.AutomaticFunctionCallingConfig(
                disable=True
            ),
            response_mime_type="application/json",
            temperature=0,
        ),
    )
    bruto = (getattr(respuesta, "text", None) or "").strip()
    if not bruto:
        raise ValueError("Gemini devolvió una respuesta vacía.")
    data = parsear_json_gemini(bruto)
    data["id_noticia"] = noticia_fuente["id_noticia"]
    if not data.get("fuente"):
        data["fuente"] = noticia_fuente["fuente"]
    if not data.get("url"):
        data["url"] = noticia_fuente["url"]
    return data


try:
    noticia_json = extraer_con_gemini(noticia)
    print("Extracción: Gemini")
except Exception as exc:
    print("No se pudo llamar a Gemini.")
    print("Motivo:", exc)
    print("Se usa el JSON de ejemplo para continuar el notebook.")
    noticia_json = JSON_EJEMPLO

noticia_json


## 11. Validar el JSON

El modelo puede omitir un campo o devolver texto en lugar de una lista.

El código debe comprobar el contrato **antes** de pensar en Obsidian. Esta función es la versión didáctica de `ValidadorJSON` del repositorio.


In [ ]:
def validar_noticia(data):
    if not isinstance(data, dict):
        raise ValueError("El documento no es un objeto JSON.")

    faltantes = [campo for campo in CAMPOS_OBLIGATORIOS if campo not in data]
    if faltantes:
        raise ValueError(f"Faltan campos: {faltantes}")

    for campo in CAMPOS_LISTA:
        if not isinstance(data[campo], list):
            raise ValueError(f"'{campo}' debe ser una lista")

    return data


noticia_validada = validar_noticia(noticia_json)
print("JSON válido. Campos presentes:")
for campo in CAMPOS_OBLIGATORIOS:
    valor = noticia_validada[campo]
    if isinstance(valor, list):
        print(f"  {campo}: lista con {len(valor)} elemento(s)")
    else:
        print(f"  {campo}: {valor!r}"[:80])


## 12. Guardar el JSON intermedio

Este archivo es la entrada que, en el proyecto del laboratorio, leería `EscritorVaultObsidian` para crear las notas Markdown.

En Colab queda en el directorio de trabajo de la sesión (`/content/`).


In [ ]:
ruta_salida = "noticia_estructurada.json"

with open(ruta_salida, "w", encoding="utf-8") as fh:
    json.dump(noticia_validada, fh, ensure_ascii=False, indent=2)
    fh.write("\n")

print("Archivo creado:", ruta_salida)
print(json.dumps(noticia_validada, ensure_ascii=False, indent=2)[:2500])


## 13. Siguiente etapa: Obsidian (no se implementa aquí)

En el repositorio, el JSON validado se convierte en una **red de notas Markdown**, no en una base de datos.

Jerarquía esperada:

```text
obsidian_vault/
|-- 00_Indice.md
|-- Noticias/N001.md
|-- Delitos/
|-- Personas/
|-- Organizaciones/
|-- Lugares/
|-- Objetos/
`-- Relaciones/
```

Cada noticia y cada entidad tiene su propia nota. Las relaciones se expresan con `[[wiki-links]]`, por ejemplo `[[La Reina]]` o `[[robo con violencia]]`.

Ese paso corresponde a `python main.py obsidian` y a la clase `EscritorVaultObsidian` (`TODO(alumno)`).


## 14. ¿Qué problemas pueden aparecer?

- El HTML se genera con JavaScript y `requests` no lo ve.
- El sitio bloquea clientes automatizados.
- El título no está en un `<h1>` o el cuerpo no está en `<article>`.
- Gemini envuelve el JSON en un bloque markdown: por eso se quitan los fences antes de `json.loads`.
- El modelo inventa una persona o un delito: hay que contrastar con el texto limpio.
- Falta `GEMINI_API_KEY`: el notebook continúa con el JSON de ejemplo.

En el laboratorio real conviene empezar con uno o dos medios conocidos (como en el notebook 01) y adaptar selectores.


## Flujo obtenido

```text
URL
 ↓
requests
 ↓
HTML
 ↓
BeautifulSoup (título + párrafos)
 ↓
Limpieza
 ↓
Diccionario intermedio
 ↓
Gemini (o JSON de ejemplo)
 ↓
Validación del contrato
 ↓
noticia_estructurada.json
 ↓
Obsidian (siguiente etapa del laboratorio)
```

Correspondencia con el proyecto:

| Este notebook | Repositorio |
| --- | --- |
| `requests` + `BeautifulSoup` | `python main.py capturar` |
| `limpiar_texto` | `LimpiadorHTML` |
| Gemini + parseo | `python main.py extraer` / `ExtractorGemini` |
| `validar_noticia` | `ValidadorJSON` |
| `noticia_estructurada.json` | `data/json/N00X.json` |
| (no implementado) | `python main.py obsidian` |
